<a href="https://colab.research.google.com/github/fidlarsyn/Introduction-Machine-Learning-with-python/blob/main/BAB_7_Working_with_Text_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Pemuatan Dataset: Sentiment Analysis of Movie Reviews**
Proses awal dalam Natural Language Processing (NLP) melibatkan pemuatan data mentah ke dalam format yang dapat diproses oleh Python. Dataset IMDb biasanya disimpan dalam folder terpisah untuk setiap kategori label.

In [ ]:
from sklearn.datasets import load_files
from sklearn.model_selection import train_test_split
import numpy as np

# Memuat dataset dari struktur direktori (pos/neg)
# categories=['pos', 'neg'] membatasi pemuatan hanya pada label tersebut
# shuffle=True memastikan data teracak agar distribusi label merata saat split
reviews_train = load_files("data/aclImdb/train/", categories=["pos", "neg"], shuffle=True)

# Mengekstrak data teks (bytes) dan label target (integer)
text_train, y_train = reviews_train.data, reviews_train.target

# Menghapus tag HTML <br /> menggunakan list comprehension untuk efisiensi
# Data mentah seringkali mengandung artifak web yang dapat dianggap sebagai noise oleh model
text_train = [doc.replace(b"<br />", b" ") for doc in text_train]

# Membagi data menjadi set pelatihan dan pengujian
# stratify=y_train memastikan proporsi kelas pos/neg tetap sama di kedua subset
# random_state=0 digunakan untuk menjamin reproduksibilitas hasil eksperimen
text_train, text_test, y_train, y_test = train_test_split(
    text_train, y_train, stratify=y_train, random_state=0
)

print(f"Jumlah sampel pelatihan: {len(text_train)}")

Penjelasan: Pembersihan tag HTML secara eksplisit sangat krusial agar model tidak mempelajari pola formatting yang tidak relevan dengan sentimen. Penggunaan stratify adalah praktik standar untuk menjaga integritas distribusi kelas pada dataset klasifikasi yang seimbang maupun timpang.


# **Representasi Teks dengan Bag-of-Words (BoW)**
Model Bag-of-Words mengubah teks menjadi vektor numerik dengan menghitung frekuensi kemunculan setiap kata, mengabaikan struktur tata bahasa dan urutan kata.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# A. Implementasi pada Toy Dataset
bards_words = ["The fool doth think he is wise,",
               "but the wise man knows himself to be a fool."]

# Inisialisasi CountVectorizer dengan parameter default
# Secara otomatis melakukan tokenisasi dan konversi ke huruf kecil (lowercase)
vect = CountVectorizer()
vect.fit(bards_words)

print(f"Ukuran kosakata: {len(vect.vocabulary_)}")
print(f"Kosakata: {vect.get_feature_names_out()}")

# B. Implementasi pada Dataset Movie Reviews
# min_df=5: Menghapus kata yang muncul di kurang dari 5 dokumen (dimensionality reduction)
# Pendekatan ini secara drastis mengurangi noise dari kata-kata langka atau typo
vect = CountVectorizer(min_df=5).fit(text_train)
X_train = vect.transform(text_train)

# Hasil transform adalah SciPy sparse matrix (Compressed Sparse Row format)
# Sangat efisien untuk data teks di mana sebagian besar entri adalah nol
print(f"Bentuk matriks X_train: {repr(X_train)}")

Penjelasan: Penggunaan min_df=5 bukan sekadar pembatasan, melainkan teknik reduksi dimensi untuk meningkatkan generalisasi model dengan mengeliminasi fitur yang terlalu spesifik (seperti nama aktor tertentu) yang tidak berguna untuk prediksi global

# **Pembersihan Data dengan Stopwords**
Stopwords adalah kata-kata fungsional yang muncul sangat sering namun memiliki muatan informasi rendah dalam konteks klasifikasi sentimen.

In [ ]:
# Menggunakan daftar stop words internal scikit-learn ('english')
# Parameter stop_words akan mengabaikan kata-kata seperti 'and', 'the', 'is'
vect = CountVectorizer(min_df=5, stop_words="english").fit(text_train)
X_train_stop = vect.transform(text_train)

print(f"Fitur setelah eliminasi stop words: {len(vect.vocabulary_)}")

Penjelasan: Menghapus stopwords dapat mempercepat proses pelatihan dan mengurangi risiko overfitting pada kata-kata umum, meskipun pada model bahasa yang lebih kompleks (seperti Deep Learning), kata-kata ini terkadang tetap dipertahankan untuk menjaga struktur sintaksis.

# **Penskalaan Data Menggunakan tf-idf**
Term Frequency-Inverse Document Frequency (tf-idf) memberikan bobot tinggi pada kata yang sering muncul dalam satu dokumen, namun jarang muncul di dokumen lain dalam korpus.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# Membangun pipeline untuk merangkai vektorisasi dan klasifikasi
# Penggunaan pipeline mencegah kebocoran data (data leakage) saat validasi silang
pipe = make_pipeline(TfidfVectorizer(min_df=5), LogisticRegression())
pipe.fit(text_train, y_train)

# Mengambil koefisien model untuk interpretasi fitur yang paling berpengaruh
# TfidfVectorizer secara internal melakukan normalisasi L2 pada vektor output
vectorizer = pipe.named_steps['tfidfvectorizer']
features = vectorizer.get_feature_names_out()
coefficients = pipe.named_steps['logisticregression'].coef_.ravel()

# Identifikasi 10 kata kunci dengan pengaruh sentimen terkuat (positif & negatif)
sorted_idx = np.argsort(coefficients)
print(f"Indikator Negatif Terkuat:\n{features[sorted_idx[:10]]}")
print(f"Indikator Positif Terkuat:\n{features[sorted_idx[-10:]]}")

Penjelasan: Penskalaan tf-idf secara otomatis meredam pengaruh kata-kata yang terlalu umum di seluruh dataset, sehingga model dapat lebih fokus pada kata-kata yang benar-benar membedakan konten antar dokumen secara unik.

# **Pengembangan Fitur dengan n-Grams**
Untuk menangkap konteks seperti negasi ("not good"), kita perlu menyertakan urutan kata yang berdekatan melalui parameter ngram_range.


In [ ]:
# ngram_range=(1, 2) mengekstrak unigram (1 kata) dan bigram (2 kata)
# Pendekatan ini menangkap konteks lokal namun meningkatkan jumlah fitur secara signifikan
vect = CountVectorizer(ngram_range=(1, 2), min_df=5).fit(text_train)
X_train_poly = vect.transform(text_train)

print(f"Jumlah fitur unigram + bigram: {len(vect.vocabulary_)}")

Visualisasi Ekspansi Fitur (Contoh: "the movie was not good"):
* Unigram saja: ["the", "movie", "was", "not", "good"] (5 fitur)
* Unigram + Bigram: ["the", "movie", "was", "not", "good", "the movie", "movie was", "was not", "not good"] (9 fitur)

Penjelasan: Meskipun n-grams meningkatkan akurasi dengan menangkap urutan kata, seorang engineer harus waspada terhadap "Kutukan Dimensi" (Curse of Dimensionality) karena jumlah fitur akan membengkak drastis, yang memerlukan memori lebih besar.

# **Tokenisasi Lanjut: Stemming dan Lemmatization**
Lemmatization menggunakan analisis linguistik untuk mengembalikan kata ke bentuk dasarnya (lemma), yang lebih akurat dibandingkan stemming sederhana.

In [ ]:
import spacy

# Memuat model bahasa spaCy dengan penanganan error
# nlp digunakan sebagai callable untuk analisis linguistik mendalam
try:
    nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])
except OSError:
    print("Model spaCy belum terinstal. Jalankan: python -m spacy download en_core_web_sm")

def custom_tokenizer(text):
    """Mengonversi dokumen menjadi daftar lemma dalam huruf kecil."""
    doc = nlp(text)
    # Mengambil token.lemma_ (bentuk dasar) dan membuang tanda baca
    return [token.lemma_.lower() for token in doc if not token.is_punct]

# Integrasi tokenizer kustom ke dalam CountVectorizer
# Parameter 'tokenizer' menerima fungsi callable yang menggantikan proses default
lemma_vect = CountVectorizer(tokenizer=custom_tokenizer, min_df=5)
X_train_lemma = lemma_vect.fit_transform(text_train)

print(f"Ukuran kosakata setelah Lemmatization: {X_train_lemma.shape[1]}")

Penjelasan: Menggunakan Lemmatization membantu mengonsolidasikan berbagai bentuk kata (seperti "runs", "running", "ran") menjadi satu fitur ("run"), namun perlu diperhatikan bahwa proses ini jauh lebih berat secara komputasi dibandingkan tokenisasi standar.

# **Topic Modeling menggunakan Latent Dirichlet Allocation (LDA)**
LDA adalah algoritma statistik tak terarah (unsupervised) yang menemukan pola topik tersembunyi dengan mengasumsikan setiap dokumen adalah campuran dari berbagai topik.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# Inisialisasi LDA pada fitur Bag-of-Words (bukan tf-idf)
# LDA adalah model probabilitas yang bekerja dengan frekuensi hitungan (counts)
# n_components=10: Menentukan jumlah topik yang ingin ditemukan
# learning_method='batch': Menggunakan seluruh data untuk update model (lebih stabil untuk dataset kecil/sedang)
lda = LatentDirichletAllocation(
    n_components=10,
    learning_method="batch",
    max_iter=25,
    random_state=0
)

# Melakukan fitting dan transformasi data teks
document_topics = lda.fit_transform(X_train)

# Menampilkan 10 kata kunci teratas untuk setiap topik
sorting = np.argsort(lda.components_, axis=1)[:, ::-1]
feature_names = vect.get_feature_names_out()

print("Ekstraksi Topik Utama:")
for i in range(10):
    top_words = [feature_names[j] for j in sorting[i, :10]]
    print(f"Topik {i}: {', '.join(top_words)}")

Penjelasan: LDA memungkinkan kita untuk mengelompokkan dokumen tanpa label secara otomatis. Dalam praktik engineering, LDA sering digunakan untuk feature discovery atau sistem rekomendasi berbasis konten dengan mengukur kemiripan distribusi topik antar dokumen.